# Reporting Bias in NYC Rodent Inspection Data

**Purpose**: Investigate whether 311 rodent complaints are a biased proxy for actual infestation,
and whether that bias is correlated with NTA median household income.

**Key question**: Are high-income NTAs inspected at higher rates *independent of actual infestation*,
because their residents file more 311 complaints? If so, the model's label
(`active_rat_signs_ind`) inherits a socioeconomic tilt.

**Data sources**:
- `features.nta_week_panel` — complaint counts, inspection counts, Active Rat Signs rate per NTA-week
- ACS 5-year estimates (B19013) — NTA-level median household income via Census tract → NTA crosswalk

**Output**: Scatterplot + findings section that feeds the `/about` page bias disclosure.

## 0. Setup

In [ ]:
import os
import warnings
from pathlib import Path

import asyncpg
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

try:
    from dotenv import load_dotenv
    load_dotenv(Path("../../.env"), override=False)
except ImportError:
    pass

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

DATABASE_URL = os.environ.get("DATABASE_URL")
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY")  # free at api.census.gov/data/key_signup.html

# Path to optional pre-downloaded ACS CSV (skips Census API if present)
ACS_CSV_PATH = Path("../../data/nta_median_income.csv")

print(f"DATABASE_URL set: {bool(DATABASE_URL)}")
print(f"CENSUS_API_KEY set: {bool(CENSUS_API_KEY)}")
print(f"ACS CSV present: {ACS_CSV_PATH.exists()}")

## 1. Load Panel Data

Aggregate `features.nta_week_panel` to NTA grain:
- **complaint rate**: mean weekly complaints per NTA (proxy for resident reporting behaviour)
- **inspection rate**: mean weekly inspections per NTA
- **infestation rate**: mean `active_rat_signs_rate` per NTA (the label signal)

We exclude weeks with zero inspections to avoid division artifacts.

In [ ]:
PANEL_SQL = """
SELECT
    nta_id,
    AVG(complaints_count)         AS mean_complaints_per_week,
    AVG(inspections_count)        AS mean_inspections_per_week,
    AVG(active_rat_signs_rate)    AS mean_infestation_rate,
    SUM(active_rat_signs_count)   AS total_active_rat_signs,
    SUM(inspections_count)        AS total_inspections,
    SUM(complaints_count)         AS total_complaints,
    COUNT(*)                      AS week_count
FROM features.nta_week_panel
WHERE inspections_count > 0
GROUP BY nta_id
HAVING SUM(inspections_count) >= 10   -- exclude NTAs with negligible inspection history
ORDER BY nta_id
"""

async def load_panel(db_url: str) -> pd.DataFrame:
    conn = await asyncpg.connect(db_url)
    try:
        rows = await conn.fetch(PANEL_SQL)
        return pd.DataFrame([dict(r) for r in rows])
    finally:
        await conn.close()

if DATABASE_URL:
    import asyncio
    panel = asyncio.get_event_loop().run_until_complete(load_panel(DATABASE_URL))
    print(f"Panel: {len(panel):,} NTAs after filtering")
    panel.head()
else:
    print("DATABASE_URL not set — skipping DB load. Set it in .env to run this notebook.")
    panel = pd.DataFrame()  # empty placeholder

## 2. Load ACS Median Household Income

Source: ACS 5-Year Estimates, Table B19013 (Median Household Income in the Past 12 Months).

We fetch tract-level estimates for all five NYC counties (FIPS: 005, 047, 061, 081, 085)
then aggregate to NTA using the 2010→2020 NTA crosswalk weights.

**Fallback**: if `data/nta_median_income.csv` exists (columns: `nta_id`, `median_income`)
it is loaded directly, skipping the Census API call.

In [ ]:
NYC_COUNTY_FIPS = ["005", "047", "061", "081", "085"]  # Bronx, Kings, New York, Queens, Richmond
NYC_STATE_FIPS = "36"
ACS_YEAR = 2022
ACS_VARIABLE = "B19013_001E"  # Median household income estimate


def _load_acs_from_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    assert {"nta_id", "median_income"}.issubset(df.columns), (
        f"{path} must have columns 'nta_id' and 'median_income'"
    )
    return df[["nta_id", "median_income"]].copy()


def _load_acs_from_api(api_key: str) -> pd.DataFrame:
    """Fetch tract-level B19013 for NYC counties from the Census API."""
    from census import Census  # noqa: PLC0415

    c = Census(api_key)
    rows = []
    for county in NYC_COUNTY_FIPS:
        data = c.acs5.state_county_tract(
            (ACS_VARIABLE, "NAME"),
            state_fips=NYC_STATE_FIPS,
            county_fips=county,
            tract="*",
            year=ACS_YEAR,
        )
        rows.extend(data)

    df = pd.DataFrame(rows)
    df["geoid"] = (
        df["state"].str.zfill(2)
        + df["county"].str.zfill(3)
        + df["tract"].str.zfill(6)
    )
    df[ACS_VARIABLE] = pd.to_numeric(df[ACS_VARIABLE], errors="coerce")
    df = df[df[ACS_VARIABLE] > 0].rename(columns={ACS_VARIABLE: "median_income"})
    return df[["geoid", "median_income"]]


def _aggregate_tracts_to_nta(tract_income: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate tract-level median income to NTA using the 2010 census tract → 2020 NTA
    crosswalk. We use a simple population-weighted mean (allocation weight as proxy).

    Crosswalk CSV expected at data/tract_to_nta_crosswalk.csv with columns:
    geoid (11-char FIPS), nta_id, weight.
    Falls back to a naive unweighted mean if the crosswalk isn't available.
    """
    crosswalk_path = Path("../../data/tract_to_nta_crosswalk.csv")
    if crosswalk_path.exists():
        xwalk = pd.read_csv(crosswalk_path, dtype={"geoid": str})
        merged = tract_income.merge(xwalk, on="geoid", how="inner")
        merged["weighted_income"] = merged["median_income"] * merged["weight"]
        return (
            merged.groupby("nta_id")
            .agg(median_income=("weighted_income", "sum"))
            .reset_index()
        )
    else:
        # Naive fallback: derive NTA prefix from tract GEOID — imprecise but usable
        # for exploratory analysis. Log a warning.
        print(
            "WARNING: tract_to_nta_crosswalk.csv not found. "
            "Using unweighted mean across tracts (exploratory only)."
        )
        # Best-effort: can't infer NTA without crosswalk — return empty
        return pd.DataFrame(columns=["nta_id", "median_income"])


if ACS_CSV_PATH.exists():
    print(f"Loading ACS data from {ACS_CSV_PATH}")
    income = _load_acs_from_csv(ACS_CSV_PATH)
elif CENSUS_API_KEY:
    print(f"Fetching ACS {ACS_YEAR} tract data from Census API …")
    tract_income = _load_acs_from_api(CENSUS_API_KEY)
    income = _aggregate_tracts_to_nta(tract_income)
    print(f"  {len(tract_income):,} tracts → {len(income):,} NTAs after crosswalk")
else:
    print(
        "Neither ACS_CSV_PATH nor CENSUS_API_KEY is available.\n"
        "Options:\n"
        "  1. Set CENSUS_API_KEY in .env (free at api.census.gov/data/key_signup.html)\n"
        "  2. Place a CSV with columns [nta_id, median_income] at data/nta_median_income.csv"
    )
    income = pd.DataFrame(columns=["nta_id", "median_income"])

print(income.head() if len(income) else "(no income data)")

## 3. Join Panel + Income, Compute Deciles

In [ ]:
if panel.empty or income.empty:
    print("Skipping join — one or both data sources are empty.")
    df = pd.DataFrame()
else:
    df = panel.merge(income, on="nta_id", how="inner")
    df["income_decile"] = pd.qcut(
        df["median_income"],
        q=10,
        labels=[f"D{i}" for i in range(1, 11)],
        duplicates="drop",
    )
    df["income_decile_int"] = df["income_decile"].cat.codes + 1

    print(f"Merged: {len(df):,} NTAs")
    print("\nIncome decile distribution:")
    print(df["income_decile"].value_counts().sort_index())

## 4. Scatterplot: Complaint Rate vs. Infestation Rate

Each point is one NTA. Color encodes median income decile (D1 = lowest income, D10 = highest).

**What to look for**:
- If the cloud tilts upward and high-income NTAs (lighter) sit in the upper-right quadrant,
  complaints are *not* a pure signal of infestation — they also reflect who files 311 reports.
- The regression line slope quantifies how much of inspection rate is explained by complaints
  versus actual infestation.

In [ ]:
if df.empty:
    print("No data to plot.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(
        "Reporting Bias in NYC Rodent Inspections\n"
        "(color = median household income decile, D1 = lowest, D10 = highest)",
        fontsize=13,
    )

    palette = sns.color_palette("RdYlGn", n_colors=10)

    # --- Left: complaint rate vs. infestation rate ---
    ax = axes[0]
    scatter = ax.scatter(
        df["mean_complaints_per_week"],
        df["mean_infestation_rate"],
        c=df["income_decile_int"],
        cmap="RdYlGn",
        vmin=1,
        vmax=10,
        alpha=0.75,
        s=60,
        edgecolors="white",
        linewidths=0.4,
    )
    # OLS trend line
    m, b = np.polyfit(df["mean_complaints_per_week"], df["mean_infestation_rate"], 1)
    x_range = np.linspace(df["mean_complaints_per_week"].min(), df["mean_complaints_per_week"].max(), 100)
    ax.plot(x_range, m * x_range + b, color="steelblue", linewidth=1.5, linestyle="--", label=f"OLS (slope={m:.3f})")
    ax.set_xlabel("Mean weekly 311 complaints per NTA")
    ax.set_ylabel("Mean Active Rat Signs rate (inspections)")
    ax.set_title("Complaint rate vs. infestation rate")
    ax.legend(fontsize=9)

    # --- Right: inspection rate vs. infestation rate ---
    ax2 = axes[1]
    ax2.scatter(
        df["mean_inspections_per_week"],
        df["mean_infestation_rate"],
        c=df["income_decile_int"],
        cmap="RdYlGn",
        vmin=1,
        vmax=10,
        alpha=0.75,
        s=60,
        edgecolors="white",
        linewidths=0.4,
    )
    m2, b2 = np.polyfit(df["mean_inspections_per_week"], df["mean_infestation_rate"], 1)
    x2 = np.linspace(df["mean_inspections_per_week"].min(), df["mean_inspections_per_week"].max(), 100)
    ax2.plot(x2, m2 * x2 + b2, color="steelblue", linewidth=1.5, linestyle="--", label=f"OLS (slope={m2:.3f})")
    ax2.set_xlabel("Mean weekly inspections per NTA")
    ax2.set_ylabel("Mean Active Rat Signs rate")
    ax2.set_title("Inspection rate vs. infestation rate")
    ax2.legend(fontsize=9)

    cbar = fig.colorbar(scatter, ax=axes, shrink=0.7, pad=0.02)
    cbar.set_label("Income decile", fontsize=10)
    cbar.set_ticks(range(1, 11))

    plt.tight_layout()
    plt.savefig("reporting_bias_scatter.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure saved to notebooks/reporting_bias_scatter.png")

## 5. Income Decile Breakdown

Box plots of complaint rate and infestation rate by income decile reveal whether
the complaint→inspection funnel is uniformly biased or concentrated at the extremes.

In [ ]:
if df.empty:
    print("No data to plot.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Distribution by Income Decile", fontsize=13)

    sns.boxplot(
        data=df, x="income_decile", y="mean_complaints_per_week",
        palette="RdYlGn", ax=axes[0], linewidth=0.8,
    )
    axes[0].set_xlabel("Income decile (D1 = lowest)")
    axes[0].set_ylabel("Mean weekly 311 complaints")
    axes[0].set_title("311 complaint rate by income decile")

    sns.boxplot(
        data=df, x="income_decile", y="mean_infestation_rate",
        palette="RdYlGn", ax=axes[1], linewidth=0.8,
    )
    axes[1].set_xlabel("Income decile (D1 = lowest)")
    axes[1].set_ylabel("Mean Active Rat Signs rate")
    axes[1].set_title("Infestation rate by income decile")

    plt.tight_layout()
    plt.show()

## 6. Correlation Analysis

In [ ]:
if df.empty:
    print("No data.")
else:
    cols = [
        "median_income",
        "mean_complaints_per_week",
        "mean_inspections_per_week",
        "mean_infestation_rate",
    ]
    corr = df[cols].corr(method="spearman")

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        corr,
        annot=True, fmt=".2f", cmap="RdBu_r", center=0,
        vmin=-1, vmax=1, linewidths=0.5, ax=ax,
    )
    ax.set_title("Spearman correlation (NTA grain)")
    plt.tight_layout()
    plt.show()

    print("\nKey correlations:")
    print(f"  income vs complaints:    {corr.loc['median_income','mean_complaints_per_week']:.3f}")
    print(f"  income vs inspections:   {corr.loc['median_income','mean_inspections_per_week']:.3f}")
    print(f"  income vs infestation:   {corr.loc['median_income','mean_infestation_rate']:.3f}")
    print(f"  complaints vs infestation: {corr.loc['mean_complaints_per_week','mean_infestation_rate']:.3f}")

## 7. Findings

*(Fill in after running with live data.)*

### What the data shows

Replace the placeholder text below with observations drawn from the actual correlation
values and scatterplots above.

**Complaint rate and income**: *(e.g., "Higher-income NTAs file more 311 complaints
per week on average — Spearman r = +0.XX — even though their infestation rates are
lower.")*

**Inspection rate and income**: *(e.g., "Inspection volume tracks complaint volume
rather than infestation rate, meaning higher-income NTAs receive more inspections
per unit of actual rat activity.")*

**Label bias**: *(e.g., "Because inspections are complaint-driven, `active_rat_signs_ind`
is observed more frequently in high-income NTAs simply because those NTAs have more
inspections. A model trained on this label will predict higher risk for high-income
areas ceteris paribus.")*

### Implications for model users

- **The label measures detected infestation, not true infestation.** An NTA with few
  inspections may have a low `active_rat_signs_rate` not because rats are absent but
  because nobody looked. Low-income NTAs with under-inspection are likely to have
  under-predicted risk scores.

- **Do not use the risk score as a resource-allocation ceiling.** The score reflects
  where DOHMH has historically inspected (complaint-driven), not where rats are most
  prevalent. Routing inspections to only high-score NTAs would reinforce the existing
  reporting bias.

- **Treat the score as a signal-boost, not ground truth.** The model is most useful
  for prioritising *within* a borough or community district — not for cross-borough
  comparisons where socioeconomic confounds are largest.

---

## ADR Reference

This notebook feeds **`docs/decisions/0003-reporting-bias.md`**.
The three bullet points in §7 are the canonical disclosure text for the `/about` page.

**What we do**: disclose the bias prominently in the UI and in the API response metadata.

**What we don't do** (Phase 1): re-weight the label, apply post-hoc calibration by income
decile, or exclude complaint features. These mitigations are deferred to Phase 3 and
require a more careful trade-off analysis.